# 04 - Churn Prediction & Explainability

**OmniStyle Customer Intelligence Platform**

This notebook builds and compares two churn models (Logistic Regression
baseline, Random Forest advanced), evaluates them with business-relevant
metrics, and uses SHAP to explain the selected model's predictions. It
answers the third and fourth levels of business intelligence: **What will
happen?** and (via SHAP + segment cross-referencing) **What should the
business do?**

### Churn definition (see `src/feature_engineering.py` docstring for full detail)

A customer is labeled **churned** if they made zero purchases in the 90-day
observation window immediately after a fixed cutoff date, having signed up
before that cutoff. Features are computed strictly from data up to the
cutoff, so there is no leakage of future information into the model.

Prerequisite:

```bash
python -m src.feature_engineering
```


In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, precision_score, recall_score, f1_score,
)

from src.config import DATA_PROCESSED_DIR, RANDOM_SEED
from src.train_model import CHURN_FEATURE_COLUMNS

np.random.seed(RANDOM_SEED)
plt.rcParams["figure.figsize"] = (8, 5)


In [ ]:
df = pd.read_csv(DATA_PROCESSED_DIR / "churn_dataset.csv")
print(df.shape)
print("Churn rate:", df["churned"].mean().round(3))
df.head()


## Train / test split

In [ ]:
X = df[CHURN_FEATURE_COLUMNS].fillna(0)
y = df["churned"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train:", X_train.shape, "Test:", X_test.shape)


## Baseline: Logistic Regression

In [ ]:
logreg = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_SEED)
logreg.fit(X_train_scaled, y_train)

logreg_pred = logreg.predict(X_test_scaled)
logreg_proba = logreg.predict_proba(X_test_scaled)[:, 1]

print(classification_report(y_test, logreg_pred, target_names=["Not Churned", "Churned"]))
print("ROC-AUC:", round(roc_auc_score(y_test, logreg_proba), 3))


## Advanced: Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=5,
    class_weight="balanced", random_state=RANDOM_SEED, n_jobs=-1,
)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, rf_pred, target_names=["Not Churned", "Churned"]))
print("ROC-AUC:", round(roc_auc_score(y_test, rf_proba), 3))


## Model comparison

In [ ]:
def summarize(name, y_true, y_pred, y_proba):
    return {
        "model": name,
        "precision": round(precision_score(y_true, y_pred), 3),
        "recall": round(recall_score(y_true, y_pred), 3),
        "f1_score": round(f1_score(y_true, y_pred), 3),
        "roc_auc": round(roc_auc_score(y_true, y_proba), 3),
    }

comparison = pd.DataFrame([
    summarize("Logistic Regression", y_test, logreg_pred, logreg_proba),
    summarize("Random Forest", y_test, rf_pred, rf_proba),
])
comparison


We select the model with the higher **ROC-AUC**, since ranking customers by
churn risk (to prioritize retention outreach) matters more for this business
use case than raw classification accuracy alone.

In [ ]:
best_name = comparison.loc[comparison["roc_auc"].idxmax(), "model"]
print("Best model:", best_name)

best_pred = rf_pred if best_name == "Random Forest" else logreg_pred
best_model = rf if best_name == "Random Forest" else logreg

fig, ax = plt.subplots(figsize=(5, 5))
cm = confusion_matrix(y_test, best_pred)
ConfusionMatrixDisplay(cm, display_labels=["Not Churned", "Churned"]).plot(ax=ax, cmap="Blues", colorbar=False)
plt.title(f"Confusion Matrix - {best_name}")
plt.tight_layout()
plt.show()


## SHAP explainability

In [ ]:
import shap

if best_name == "Random Forest":
    explainer = shap.TreeExplainer(rf)
    shap_values = explainer.shap_values(X_test)
    if isinstance(shap_values, list):
        shap_values_pos = shap_values[1]
    elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
        shap_values_pos = shap_values[:, :, 1]
    else:
        shap_values_pos = shap_values

    shap.summary_plot(shap_values_pos, X_test, show=True)
else:
    print("Best model is Logistic Regression; SHAP TreeExplainer is skipped in this notebook. "
          "See coefficients below instead.")
    print(pd.Series(logreg.coef_[0], index=CHURN_FEATURE_COLUMNS).sort_values())


## Global feature importance (bar chart)

In [ ]:
if best_name == "Random Forest":
    mean_abs_shap = pd.DataFrame({
        "feature": X_test.columns,
        "mean_abs_shap": np.abs(shap_values_pos).mean(axis=0),
    }).sort_values("mean_abs_shap", ascending=False)

    mean_abs_shap.head(10).plot(kind="barh", x="feature", y="mean_abs_shap", figsize=(8, 6), legend=False, color="indianred")
    plt.gca().invert_yaxis()
    plt.title("Top 10 Churn Drivers (Mean |SHAP value|)")
    plt.xlabel("Mean |SHAP value|")
    plt.tight_layout()
    plt.show()

    mean_abs_shap.head(10)


## What this means for the business

- Features with the highest mean |SHAP value| are the strongest drivers of
  the model's churn predictions. In practice, `recency_days`, `frequency`,
  and `monetary`-related features tend to dominate, meaning **how recently
  and how often a customer buys** is the clearest early warning signal —
  more so than demographic attributes like age or gender.
- `average_discount` also tends to matter: customers who only respond to
  heavy discounting show different churn dynamics than full-price shoppers,
  which should inform how aggressively marketing discounts to retain them.
- Because SHAP values are additive and per-customer, the same model can
  generate a **local explanation for any individual customer** — useful for
  a retention agent who wants to know *why* a specific customer is flagged
  as high-risk before reaching out.

See `reports/business_recommendations.md` (generated by
`python -m src.train_model`) for the full, data-driven recommendations.
